## Silver: dim_bundesland.

Reference dimension: one row per Bundesland (16 rows + offshore = 17 total),
combining tax contribution and population. Used as the basis for Flow B
allocation in gold.

Sources:
  bronze.bmf_steueraufkommen_bund_2024
  bronze.destatis_bevoelkerung_2024

Output columns:
  bundesland             : canonical Bundesland name (matches dim_anlage)
  bund_anteil_mio_eur    : federal tax revenue contribution 2024 (BMF Übersicht 5a)
  einwohner              : population at year-end 2024 (Destatis)
  bund_anteil_share      : share of total federal revenue (computed)


In [0]:
from pyspark.sql import functions as F


steuern = (
    spark.table("eeg_dev.bronze.bmf_steueraufkommen_bund_2024")
    .select(
        F.col("bundesland"),
        F.col("bund_anteil_mio_eur_2024").cast("double").alias("bund_anteil_mio_eur"),
    )
)

# ADJUST column names below to match what's actually in destatis_bevoelkerung_2024.
# After running step 3, replace 'bundesland_destatis' and 'einwohner_2024' with the
# real column names from the Destatis CSV.
bev = (
    spark.table("eeg_dev.bronze.destatis_bevoelkerung_2024")
    .select(
        F.col("Bundesland").alias("bundesland_dest"),
        F.col("einwohner_2024").cast("long").alias("einwohner"),
    )
)

# Outer join to keep both tables aligned + offshore as a synthetic row
combined = (
    steuern.alias("s")
    .join(bev.alias("b"), F.col("s.bundesland") == F.col("b.bundesland_dest"), how="outer")
    .select(
        F.coalesce(F.col("s.bundesland"), F.col("b.bundesland_dest")).alias("bundesland"),
        F.col("bund_anteil_mio_eur"),
        F.col("einwohner"),
    )
)

# Compute the share for Flow B allocation
total_steuern = combined.agg(F.sum("bund_anteil_mio_eur")).collect()[0][0]
print(f"Total Bund-Anteil 2024: {total_steuern:,.0f} Mio EUR")

dim = combined.withColumn(
    "bund_anteil_share",
    F.col("bund_anteil_mio_eur") / F.lit(total_steuern),
)

# Append a synthetic offshore row so the table aligns with dim_anlage's geography
offshore = spark.createDataFrame(
    [("Ausschließliche Wirtschaftszone", None, None, None)],
    schema=dim.schema,
)
dim_with_offshore = dim.unionByName(offshore)

(
    dim_with_offshore.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("eeg_dev.silver.dim_bundesland")
)

print("\ndim_bundesland written:")
spark.table("eeg_dev.silver.dim_bundesland").orderBy(F.desc("bund_anteil_mio_eur")).show(20, truncate=False)

In [0]:
%sql
SELECT SUM(einwohner_2024) AS total FROM eeg_dev.bronze.destatis_bevoelkerung_2024;